# p3-v81 — Fine-tuning BETO + Ensemble (7 modelos) con pipeline histórico


**Pipeline completo:** limpieza histórica → 29 features → fine-tuning BETO + RoBERTa-BNE → 7 modelos → ensemble voting

**Modelos:**
1. BETO fine-tuneado (desde `dccuchile/bert-base-spanish-wwm-cased`)
2. RoBERTa-BNE fine-tuneado (desde `PlanTL-GOB-ES/roberta-base-bne`)
3. CharCNN (vocab histórico, text_base)
4. MLP (embeddings 768d + features 29d)
5. TF-IDF char_wb + LogReg
6. LinearSVC calibrado
7. Regresión ordinal (Ridge)


In [1]:

# === PASO 1 — Imports y configuración unificada ===
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import re, gc, math, random, json, itertools, warnings
from datetime import datetime
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.special import softmax

from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

from datasets import Dataset as HFDataset
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments,
)

import nltk
from nltk.corpus import stopwords
import joblib

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

# === Paths: editar para Kaggle ===
# Kaggle: ROOT = Path('/kaggle/working'), DATA_DIR = Path('/kaggle/input/<dataset>')
ROOT = Path(".").resolve()
DATA_DIR = Path("./data")

BETO_NAME = "dccuchile/bert-base-spanish-wwm-cased"
MAX_LENGTH = 384
EVAL_STRIDE = 128
BATCH_SIZE = 32

# BETO fine-tuning
BETO_FT_BATCH_SIZE = 16
BETO_FT_LR = 2e-5
BETO_FT_EPOCHS = 4
BETO_FT_CKPT = str(ROOT / "ckpt-beto-best")

# RoBERTa-BNE
ROBERTA_BNE_NAME = "PlanTL-GOB-ES/roberta-base-bne"
ROBERTA_MAX_LENGTH = 512
ROBERTA_EVAL_STRIDE = 128
ROBERTA_BATCH_SIZE = 16
ROBERTA_EPOCHS = 4
ROBERTA_LR = 1.5e-5
ROBERTA_BNE_CKPT = str(ROOT / "ckpt-roberta-bne-best")

# Umbrales de calidad
UMBRAL_CHARS = 20
UMBRAL_WORDS = 3
UMBRAL_ALPHA = 0.3
UMBRAL_OCR_CRIT = 0.8

# NLTK: descarga local
for res_id, res_path in [("punkt", "tokenizers/punkt"), ("stopwords", "corpora/stopwords"),
                         ("punkt_tab", "tokenizers/punkt_tab")]:
    try:
        nltk.data.find(res_path)
    except LookupError:
        nltk.download(res_id, quiet=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| GPUs visibles:", torch.cuda.device_count())
print("Configuración lista.")


device: cuda | GPUs visibles: 1
Configuración lista.


## PASO 2 — Funciones de limpieza histórica


In [2]:

# === PASO 2 — Pipeline de limpieza ===

# --- Paso 1: UTF-8 validation ---
def validate_utf8_series(series):
    records = []
    for raw in series:
        if not isinstance(raw, str):
            s = str(raw) if raw is not None else ""
            records.append({"raw_text": s, "text_utf8": s,
                            "encoding_source": "coerced", "decode_flag": 0})
            continue
        try:
            raw.encode("utf-8").decode("utf-8")
            records.append({"raw_text": raw, "text_utf8": raw,
                            "encoding_source": "utf-8", "decode_flag": 1})
        except (UnicodeDecodeError, UnicodeEncodeError):
            try:
                text = raw.encode("latin1").decode("utf-8", errors="replace")
                records.append({"raw_text": raw, "text_utf8": text,
                                "encoding_source": "latin1->utf-8", "decode_flag": 0})
            except Exception:
                text = raw.encode("utf-8", errors="replace").decode("utf-8")
                records.append({"raw_text": raw, "text_utf8": text,
                                "encoding_source": "forced_replace", "decode_flag": 0})
    return pd.DataFrame(records)

# --- Paso 2: Doble vista (text_base / text_norm_soft) ---
def normalize_whitespace_soft(text):
    text = re.sub(r"[ 	]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

# --- Paso 3: Strip control chars ---
def strip_control_chars(text):
    chars = []
    for ch in text:
        cp = ord(ch)
        if ch in ("\n", "\t"):
            chars.append(ch)
        elif cp < 0x20 or cp == 0x7F:
            continue
        else:
            chars.append(ch)
    return "".join(chars)

# --- Paso 4: Resolver guiones de corte ---
HYPHEN_PATTERN = re.compile(
    r"([a-zA-ZáéíóúüñÁÉÍÓÚÜÑſf]{2,})-\n([a-zA-ZáéíóúüñÁÉÍÓÚÜÑ]{2,})"
)

def resolve_linebreak_hyphens(text):
    matches = list(HYPHEN_PATTERN.finditer(text))
    count = len(matches)
    dehyphenated = HYPHEN_PATTERN.sub(r"\1\2", text)
    return dehyphenated, count

# --- Paso 5: Detección editorial ---
PATTERN_CITATION_5 = re.compile(
    r"(?:lib|cap|pag|fol|epist|tom|vol|num|pág|col|par|p\.|cap\.|lib\.|fol\.|epist\.|tom\.|vol\.)"
    r"\.?\s*\d+(?:[.,]\s*\d+)*",
    re.IGNORECASE
)
PATTERN_MARGINAL_NUM = re.compile(r"^\s*\(?\d+\)?\s+", re.MULTILINE)
PATTERN_DENSE_PUNCT = re.compile(r"(?:[.,;:]{4,})")

def detect_editorial_noise(text):
    citations = list(PATTERN_CITATION_5.finditer(text))
    marginal = list(PATTERN_MARGINAL_NUM.finditer(text))
    dense_punct = list(PATTERN_DENSE_PUNCT.finditer(text))
    spans = [(m.start(), m.end(), "citation") for m in citations]
    spans += [(m.start(), m.end(), "marginal_num") for m in marginal]
    spans += [(m.start(), m.end(), "dense_punct") for m in dense_punct]
    spans.sort()
    citation_count = len(citations)
    numeric_ref_count = sum(1 for m in marginal)
    total_chars = max(len(text), 1)
    numeric_ref_density = (numeric_ref_count * 100) / total_chars if total_chars else 0
    return {
        "editorial_spans": spans,
        "citation_count": citation_count,
        "citation_flag": int(citation_count > 0 or numeric_ref_count > 0),
        "numeric_reference_density": round(numeric_ref_density, 6),
    }

# --- Paso 6: OCR noise scoring ---
SUSPECT_CHARS = set("^|~`@#$%^&*_={}[]\\|<>")
ALNUM_MIXED_PATTERN = re.compile(r"\b(?=[a-zA-Z]*\d)(?=\d*[a-zA-Z])[a-zA-Z0-9]+\b")
WEIRD_SEQUENCE_PATTERN = re.compile(
    r'[^\w\s\u00e1\u00e9\u00ed\u00f3\u00fa\u00fc\u00f1\u00c1\u00c9\u00cd\u00d3\u00da\u00dc\u00d1\.,;:\-\"\'\u00ab\u00bb\(\)\n\t]{2,}'
)

def score_ocr_noise(text):
    if not text or len(text) == 0:
        return {"ocr_noise_score": 1.0, "weird_char_rate": 1.0,
                "alnum_mixed_token_rate": 0.0, "suspect_tokens_list": []}
    total_chars = len(text)
    total_tokens = len(text.split())
    weird_chars = sum(1 for c in text if c in SUSPECT_CHARS)
    weird_char_rate = weird_chars / max(total_chars, 1)
    weird_seq_matches = list(WEIRD_SEQUENCE_PATTERN.findall(text))
    alnum_mixed = list(ALNUM_MIXED_PATTERN.findall(text))
    alnum_rate = len(alnum_mixed) / max(total_tokens, 1)
    suspect_tokens = [t for t in text.split() if
                      any(c in SUSPECT_CHARS for c in t) or
                      (len(t) > 1 and any(c.isdigit() for c in t) and any(c.isalpha() for c in t))]
    ocr_score = min(1.0, weird_char_rate * 10 + alnum_rate * 3 + len(weird_seq_matches) / max(total_tokens, 1) * 5)
    return {
        "ocr_noise_score": round(ocr_score, 6),
        "weird_char_rate": round(weird_char_rate, 6),
        "alnum_mixed_token_rate": round(alnum_rate, 6),
        "suspect_tokens_list": suspect_tokens[:20],
    }

print("Funciones de limpieza listas.")


Funciones de limpieza listas.


## PASO 3 — Funciones de features + aumentación


In [3]:

# === PASO 3 — Ingeniería de características + aumentación ===

# --- Paso 7: Estilometría (12 features) ---
def extract_stylometry(text):
    if not text:
        return {k: 0.0 for k in ["semicolon_rate", "emdash_rate", "colon_rate",
                "paren_rate", "quote_angle_rate", "excl_rate", "quest_rate",
                "comma_rate", "asterisk_rate", "dash_rate", "quote_double_rate",
                "sentence_len_mean"]}
    n = max(len(text), 1)
    words = text.split()
    nw = max(len(words), 1)
    sents = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    sent_lens = [len(s.split()) for s in sents] if sents else [nw]
    return {
        "semicolon_rate": text.count(";") * 100 / n,
        "emdash_rate": (text.count("—") + text.count("–")) * 100 / n,
        "colon_rate": text.count(":") * 100 / n,
        "paren_rate": (text.count("(") + text.count(")")) * 100 / n,
        "quote_angle_rate": (text.count("«") + text.count("»")) * 100 / n,
        "excl_rate": text.count("!") * 100 / n,
        "quest_rate": text.count("?") * 100 / n,
        "comma_rate": text.count(",") * 100 / n,
        "asterisk_rate": text.count("*") * 100 / n,
        "dash_rate": text.count("-") * 100 / n,
        "quote_double_rate": text.count("") * 100 / n,
        "sentence_len_mean": round(np.mean(sent_lens), 4),
    }

# --- Paso 8: Ortografía histórica (4 features) ---
LATIN_ABBREV_PATTERN = re.compile(
    r"\b(?:lib|cap|pag|fol|epist|tom|vol|num|pág|col|par|ss|ib|id|op|cit|et|ca|vs)\.",
    re.IGNORECASE
)
HONORIFIC_PATTERN = re.compile(
    r"\b(?:D\.|Don|Doña|Exc\.|S\.M\.|S\.A\.|R\.P\.|V\.E\.|V\.S\.|Fray|Sor|Sto|Sta)\.?",
)
LONG_S_MARKERS = re.compile(r"ſ")  # ſ

def extract_historical_spelling(text, decade=None):
    n = max(len(text), 1)
    nw = max(len(text.split()), 1)
    long_s_count = len(LONG_S_MARKERS.findall(text))
    obsolete_accent = sum(1 for c in text if c in "ÁÉÍÓÚ")
    latin_abbrevs = len(LATIN_ABBREV_PATTERN.findall(text))
    honorific_caps = len(HONORIFIC_PATTERN.findall(text))
    return {
        "obsolete_accent_rate": obsolete_accent * 100 / n,
        "long_s_proxy_rate": long_s_count * 100 / n,
        "latin_abbrev_rate": latin_abbrevs * 100 / nw,
        "honorific_caps_rate": honorific_caps * 100 / nw,
    }

# --- Paso 9: OCR features (3 features) ---
spanish_stops = set(stopwords.words("spanish"))
modern_lexicon_base = {
    "el", "la", "los", "las", "de", "del", "en", "un", "una", "y", "e", "o",
    "a", "ante", "bajo", "con", "contra", "para", "por", "que", "su", "le",
    "se", "no", "es", "lo", "como", "más", "pero", "si", "este", "esta", "entre"
}

def extract_ocr_features(text, modern_lexicon=None):
    if modern_lexicon is None:
        modern_lexicon = set()
    nw = max(len(text.split()), 1)
    tokens = text.split()
    non_alpha = sum(1 for t in tokens if not re.match(
        r"^[a-zA-ZáéíóúüñÁÉÍÓÚÜÑ]+$", t))
    oov = sum(1 for t in tokens if t.lower() not in modern_lexicon)
    corrupt = sum(1 for t in tokens if re.search(
        r"[^\w\sáéíóúüñÁÉÍÓÚÜÑ]", t))
    return {
        "ocr_symbol_rate": non_alpha * 100 / nw,
        "oov_rate": oov * 100 / nw,
        "corrupt_token_rate": corrupt * 100 / nw,
    }

# --- Paso 10: Sintaxis y frecuencia léxica (5 features) ---
SUBORDINATORS = {"que", "pues", "donde", "si", "como", "cuando",
               "aunque", "porque", "mientras", "cual", "quien", "cuyo"}
STOPWORDS_HISTORIC = {"á", "él", "ella", "ello", "ellos", "dixo",
                      "della", "dello", "dellas", "dellos", "deste", "desta",
                      "assi", "ansí", "ansi", "aunque", "pues", "ca", "mas",
                      "porque", "sobre", "entre", "cabe", "según", "sin", "con"}

def extract_syntactic_features(text):
    nw = max(len(text.split()), 1)
    tokens = text.lower().split()
    ttr = len(set(tokens)) / nw if tokens else 0
    subord_count = sum(1 for t in tokens if t in SUBORDINATORS)
    subordination_rate = subord_count * 100 / nw
    historic_stops = sum(1 for t in tokens if t in STOPWORDS_HISTORIC)
    historic_stopword_rate = historic_stops * 100 / nw
    clauses = re.split(r"[,.;:!?]+", text)
    clause_lens = [len(c.split()) for c in clauses if c.strip()]
    clause_len_mean = np.mean(clause_lens) if clause_lens else nw
    citation_pat = len(re.findall(
        r"[A-Z][a-záéíóúüñ]+\s+(?:lib|cap|pag|fol|epist)\.\s*\d+", text))
    return {
        "ttr": round(ttr, 4),
        "subordination_rate": round(subordination_rate, 4),
        "historic_stopword_rate": round(historic_stopword_rate, 4),
        "clause_len_mean": round(clause_len_mean, 4),
        "citation_pattern_rate": citation_pat * 100 / nw,
    }

FEATURE_COLUMNS = [
    "semicolon_rate", "emdash_rate", "colon_rate", "paren_rate",
    "quote_angle_rate", "excl_rate", "quest_rate", "comma_rate",
    "asterisk_rate", "dash_rate", "quote_double_rate", "sentence_len_mean",
    "obsolete_accent_rate", "long_s_proxy_rate", "latin_abbrev_rate",
    "honorific_caps_rate", "ocr_symbol_rate", "oov_rate", "corrupt_token_rate",
    "ttr", "subordination_rate", "historic_stopword_rate", "clause_len_mean",
    "citation_pattern_rate", "numeric_reference_density", "ocr_noise_score",
    "weird_char_rate", "alnum_mixed_token_rate", "count_linebreak_hyphen",
]

# --- Augmentación (Pasos 15-18) ---
OCR_CONFUSIONS = {
    "s": ["f", "ſ", "s"],
    "S": ["F", "S"],
    "u": ["v", "u"],
    "U": ["V", "U"],
    "v": ["u", "v"],
    "V": ["U", "V"],
    "i": ["j", "i"],
    "I": ["J", "I"],
    "j": ["i", "j"],
    "c": ["e", "c"],
    "C": ["E", "C"],
    "e": ["c", "e"],
    "E": ["C", "E"],
    "n": ["m", "n"],
    "m": ["n", "m"],
}

def augment_ocr_noise(text, prob=0.03):
    chars = list(text)
    for i in range(len(chars)):
        if np.random.random() < prob and chars[i] in OCR_CONFUSIONS:
            chars[i] = np.random.choice(OCR_CONFUSIONS[chars[i]])
    return "".join(chars)

def simulate_hyphenation(text, prob=0.1):
    tokens = text.split()
    new_tokens = []
    for t in tokens:
        if len(t) >= 8 and np.random.random() < prob:
            split_point = len(t) // 2
            new_tokens.append(t[:split_point] + "-\n" + t[split_point:])
        else:
            new_tokens.append(t)
    return " ".join(new_tokens)

PROTECTED_TOKENS = {
    "lib.", "cap.", "pag.", "fol.", "epist.", "tom.", "vol.",
    "D.", "Don", "Doña", "Exc.", "S.M.", "S.A.",
    "ſ", "ſe", "ſu", "poreft", "della", "dello",
}

def historical_masking(text, mask_prob=0.08):
    tokens = text.split()
    masked = []
    for t in tokens:
        if t in PROTECTED_TOKENS:
            masked.append(t)
        elif np.random.random() < mask_prob and len(t) > 3:
            masked.append("[MASK]")
        else:
            masked.append(t)
    return " ".join(masked)

HISTORIC_SYNONYMS = {
    "dice": ["dize", "dice", "dixe"],
    "hacer": ["hazer", "hacer", "facer"],
    "mujer": ["muger", "mujer"],
    "decir": ["dezir", "decir", "dexir"],
    "así": ["assi", "ansí", "ansi", "así"],
    "mismo": ["mismo", "mesmo"],
    "escribir": ["escrevir", "escribir"],
    "cosa": ["cosa", "cossa"],
    "año": ["año", "anno"],
    "tiempo": ["tiempo", "tienpo", "tyempo"],
    "autor": ["autor", "authór", "auctor"],
    "parecer": ["parecer", "parescer"],
    "razón": ["razón", "raçón", "razon"],
    "hombre": ["hombre", "ombre"],
}

def restricted_lexical_augment(text, prob=0.1):
    tokens = text.split()
    augmented = []
    changed = False
    for t in tokens:
        t_lower = t.lower()
        if t_lower in HISTORIC_SYNONYMS and np.random.random() < prob:
            options = HISTORIC_SYNONYMS[t_lower]
            chosen = np.random.choice(options)
            if t[0].isupper():
                chosen = chosen.capitalize()
            augmented.append(chosen)
            changed = True
        else:
            augmented.append(t)
    return " ".join(augmented), changed

print(f"Funciones de features listas. {len(FEATURE_COLUMNS)} features definidas.")


Funciones de features listas. 29 features definidas.


## PASO 4 — Cargar CSVs + pipeline de limpieza


In [4]:

# === PASO 4 — Cargar CSVs + ejecutar pipeline de limpieza ===

train_raw = pd.read_csv(DATA_DIR / "train.csv", engine="python", quotechar='"', on_bad_lines="warn")
eval_raw  = pd.read_csv(DATA_DIR / "eval.csv",  engine="python", quotechar='"', on_bad_lines="warn")

# --- Paso 1: UTF-8 validation ---
df_utf8 = validate_utf8_series(train_raw["text"])
train = pd.concat([df_utf8, train_raw.drop(columns=["text"])], axis=1)

eval_utf8 = validate_utf8_series(eval_raw["text"])
eval_df = pd.concat([eval_utf8, eval_raw.drop(columns=["text"])], axis=1)

print(f"UTF-8 OK train: {train['decode_flag'].sum()} / {len(train)}")
print(f"UTF-8 OK eval:  {eval_df['decode_flag'].sum()} / {len(eval_df)}")

# --- Paso 2: Doble vista ---
train["text_raw"] = train["raw_text"]
train["text_base"] = train["text_utf8"]
train["text_norm_soft"] = train["text_utf8"].apply(normalize_whitespace_soft)

eval_df["text_raw"] = eval_df["raw_text"]
eval_df["text_base"] = eval_df["text_utf8"]
eval_df["text_norm_soft"] = eval_df["text_utf8"].apply(normalize_whitespace_soft)

# --- Paso 3: Strip control chars ---
train["text_base"] = train["text_base"].apply(strip_control_chars)
train["text_norm_soft"] = train["text_norm_soft"].apply(strip_control_chars)
train["text_norm_soft"] = train["text_norm_soft"].str.replace("\t", " ")
train["text_norm_soft"] = train["text_norm_soft"].apply(lambda t: re.sub(r" +", " ", t))

eval_df["text_base"] = eval_df["text_base"].apply(strip_control_chars)
eval_df["text_norm_soft"] = eval_df["text_norm_soft"].apply(strip_control_chars)
eval_df["text_norm_soft"] = eval_df["text_norm_soft"].str.replace("\t", " ")
eval_df["text_norm_soft"] = eval_df["text_norm_soft"].apply(lambda t: re.sub(r" +", " ", t))

# --- Paso 4: Resolver guiones de corte ---
hyphen_res = train["text_base"].apply(resolve_linebreak_hyphens)
train["text_dehyphenated"] = [r[0] for r in hyphen_res]
train["count_linebreak_hyphen"] = [r[1] for r in hyphen_res]
train["had_linebreak_hyphen"] = (train["count_linebreak_hyphen"] > 0).astype(int)

hyphen_res_e = eval_df["text_base"].apply(resolve_linebreak_hyphens)
eval_df["text_dehyphenated"] = [r[0] for r in hyphen_res_e]
eval_df["count_linebreak_hyphen"] = [r[1] for r in hyphen_res_e]
eval_df["had_linebreak_hyphen"] = (eval_df["count_linebreak_hyphen"] > 0).astype(int)

print(f"Train guiones de corte: {train['had_linebreak_hyphen'].sum()} muestras")

# --- Paso 5: Detección editorial ---
ed_res = train["text_base"].apply(detect_editorial_noise)
train["citation_count"] = [r["citation_count"] for r in ed_res]
train["citation_flag"] = [r["citation_flag"] for r in ed_res]
train["numeric_reference_density"] = [r["numeric_reference_density"] for r in ed_res]

ed_res_e = eval_df["text_base"].apply(detect_editorial_noise)
eval_df["citation_count"] = [r["citation_count"] for r in ed_res_e]
eval_df["citation_flag"] = [r["citation_flag"] for r in ed_res_e]
eval_df["numeric_reference_density"] = [r["numeric_reference_density"] for r in ed_res_e]

# --- Paso 6: OCR noise scoring ---
ocr_res = train["text_base"].apply(score_ocr_noise)
train["ocr_noise_score"] = [r["ocr_noise_score"] for r in ocr_res]
train["weird_char_rate"] = [r["weird_char_rate"] for r in ocr_res]
train["alnum_mixed_token_rate"] = [r["alnum_mixed_token_rate"] for r in ocr_res]

ocr_res_e = eval_df["text_base"].apply(score_ocr_noise)
eval_df["ocr_noise_score"] = [r["ocr_noise_score"] for r in ocr_res_e]
eval_df["weird_char_rate"] = [r["weird_char_rate"] for r in ocr_res_e]
eval_df["alnum_mixed_token_rate"] = [r["alnum_mixed_token_rate"] for r in ocr_res_e]

# --- Features de estilometría (Paso 7) ---
stylo_df = train["text_base"].apply(lambda t: pd.Series(extract_stylometry(t)))
train = pd.concat([train, stylo_df], axis=1)
stylo_df_e = eval_df["text_base"].apply(lambda t: pd.Series(extract_stylometry(t)))
eval_df = pd.concat([eval_df, stylo_df_e], axis=1)

# --- Features de ortografía histórica (Paso 8) ---
hist_df = train["text_base"].apply(lambda t: pd.Series(extract_historical_spelling(t)))
train = pd.concat([train, hist_df], axis=1)
hist_df_e = eval_df["text_base"].apply(lambda t: pd.Series(extract_historical_spelling(t)))
eval_df = pd.concat([eval_df, hist_df_e], axis=1)

# --- Lexicon para OCR features (basado en train) ---
all_words = " ".join(train["text_base"]).lower().split()
word_freq = Counter(all_words)
historical_lexicon = set(w for w, c in word_freq.most_common(20000))
modern_lexicon = historical_lexicon | spanish_stops | modern_lexicon_base

# --- Features OCR (Paso 9) ---
ocr_feat_df = train["text_base"].apply(lambda t: pd.Series(extract_ocr_features(t, modern_lexicon)))
train = pd.concat([train, ocr_feat_df], axis=1)
ocr_feat_df_e = eval_df["text_base"].apply(lambda t: pd.Series(extract_ocr_features(t, modern_lexicon)))
eval_df = pd.concat([eval_df, ocr_feat_df_e], axis=1)

# --- Features sintácticas (Paso 10) ---
syn_df = train["text_base"].apply(lambda t: pd.Series(extract_syntactic_features(t)))
train = pd.concat([train, syn_df], axis=1)
syn_df_e = eval_df["text_base"].apply(lambda t: pd.Series(extract_syntactic_features(t)))
eval_df = pd.concat([eval_df, syn_df_e], axis=1)

print("Pipeline de limpieza y features completo.")


UTF-8 OK train: 31403 / 31403
UTF-8 OK eval:  3490 / 3490
Train guiones de corte: 0 muestras
Pipeline de limpieza y features completo.


## PASO 5 — Quality tiers + GroupKFold split


In [5]:

# === PASO 5 — Label encoding + quality tiers + GroupKFold ===

def assign_quality_tier(row):
    if row["ocr_noise_score"] > UMBRAL_OCR_CRIT or row["num_chars"] < UMBRAL_CHARS or row["alpha_ratio"] < UMBRAL_ALPHA:
        return "discard"
    elif row["ocr_noise_score"] > 0.3 or row["alpha_ratio"] < 0.5:
        return "medium"
    else:
        return "high"

train["num_chars"] = train["text_base"].str.len()
train["num_words"] = train["text_base"].str.split().str.len()
train["alpha_ratio"] = train["text_base"].apply(
    lambda t: sum(1 for c in t if c.isalpha()) / max(len(t), 1))

train = train.dropna(subset=["decade"]).copy()
train["decade"] = train["decade"].astype(int)

train["quality_tier"] = train.apply(assign_quality_tier, axis=1)
print(f"Quality tiers:\n{train['quality_tier'].value_counts().to_string()}")

# Filtrar discard
train = train[train["quality_tier"] != "discard"].copy()

# Label encoding (keys str para HuggingFace new validation)
labels = sorted(train["decade"].unique().tolist())
label2id = {str(d): i for i, d in enumerate(labels)}
id2label = {i: str(d) for i, d in enumerate(labels)}
num_labels = len(labels)
train["label"] = train["decade"].astype(str).map(label2id).astype(int)

# Drop duplicates sobre text_base
train = train.drop_duplicates(subset=["text_base", "decade"]).reset_index(drop=True)
print(f"Después de filtro: {len(train)} muestras, num_labels: {num_labels}")

# Soft labels ordinales
def make_soft_label_matrix(num_labels, sigma=1.5):
    idx = np.arange(num_labels)
    M = np.zeros((num_labels, num_labels), dtype=np.float32)
    for k in range(num_labels):
        w = np.exp(-((idx - k) ** 2) / (2.0 * sigma ** 2))
        M[k] = w / w.sum()
    return M

SOFT_TARGETS = make_soft_label_matrix(num_labels, 1.5)
SOFT_TARGETS_T = torch.tensor(SOFT_TARGETS, dtype=torch.float32)

# GroupKFold split
train["source_doc_id"] = train["text_base"].str[:50].apply(hash) % 10000
train["source_doc_id"] = train["source_doc_id"].abs()

vc = train["label"].value_counts()
strat_eligible = train["label"].isin(vc[vc >= 2].index)
pool = train[strat_eligible].reset_index(drop=True)
singletons = train[~strat_eligible].reset_index(drop=True)

gkf = GroupKFold(n_splits=5)
groups = pool["source_doc_id"]

for fold, (train_idx, val_idx) in enumerate(gkf.split(pool, pool["label"], groups)):
    train_df = pool.iloc[train_idx].reset_index(drop=True)
    val_df = pool.iloc[val_idx].reset_index(drop=True)
    break  # fold 0

train_df = pd.concat([train_df, singletons], ignore_index=True)
train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

val_true_lbl = val_df["label"].to_numpy()
val_true_dec = val_df["decade"].to_numpy()

train_docs = set(train_df["source_doc_id"])
val_docs = set(val_df["source_doc_id"])
overlap = train_docs & val_docs
print(f"train/val: {len(train_df)}/{len(val_df)}  num_labels: {num_labels}")
print(f"Documentos solapados (debe ser 0): {len(overlap)}")

# Auxiliares
def align_proba(proba, classes_, num_labels):
    out = np.zeros((proba.shape[0], num_labels), dtype=np.float32)
    for j, c in enumerate(classes_):
        out[:, int(c)] = proba[:, j]
    return out

def report(name, val_proba):
    pred = np.argmax(val_proba, axis=-1)
    acc = accuracy_score(val_true_lbl, pred)
    mae = mean_absolute_error(val_true_dec, [labels[i] for i in pred])
    print(f"  [{name:32s}] val acc={acc:.4f}  mae={mae:.4f}")
    return acc

# Eval: sin filtrar, mantener id original
eval_df = eval_df[eval_df["text_base"].str.len() > 0].copy().reset_index(drop=True)
print(f"Eval después de limpieza: {len(eval_df)} muestras")


Quality tiers:
quality_tier
high       28448
medium      2410
discard      545
Después de filtro: 30825 muestras, num_labels: 39
train/val: 24660/6165  num_labels: 39
Documentos solapados (debe ser 0): 0
Eval después de limpieza: 3490 muestras


## PASO 6 — Aumentación sobre train_df


In [6]:

# === PASO 6 — Aumentación sobre train_df ===

aug_rows = []

# 1. OCR simulation sobre quality_tier == "medium"
df_medium = train_df[train_df["quality_tier"] == "medium"].copy()
if len(df_medium) > 0:
    aug_ocr = df_medium.copy()
    aug_ocr["text_base"] = aug_ocr["text_base"].apply(lambda t: augment_ocr_noise(t, prob=0.05))
    aug_ocr["aug_type"] = "ocr_simulated"
    aug_rows.append(aug_ocr)
    print(f"Aumentación OCR: {len(aug_ocr)} filas")

# 2. Hyphenation simulation sobre quality_tier == "high"
df_high = train_df[train_df["quality_tier"] == "high"].copy()
if len(df_high) > 0:
    aug_hyphen = df_high.copy()
    aug_hyphen["text_base"] = aug_hyphen["text_base"].apply(
        lambda t: simulate_hyphenation(t, prob=0.15))
    aug_hyphen["aug_type"] = "hyphenation"
    aug_rows.append(aug_hyphen)
    print(f"Aumentación hyphen: {len(aug_hyphen)} filas")

# 3. Historical masking sobre quality_tier == "high"
if len(df_high) > 0:
    aug_masked = df_high.copy()
    aug_masked["text_base"] = aug_masked["text_base"].apply(historical_masking)
    aug_masked["aug_type"] = "masked"
    aug_rows.append(aug_masked)
    print(f"Aumentación masking: {len(aug_masked)} filas")

# 4. Lexical augmentation sobre quality_tier == "high"
if len(df_high) > 0:
    lex_results = df_high["text_base"].apply(restricted_lexical_augment)
    lex_texts = [r[0] for r in lex_results]
    lex_changed = [r[1] for r in lex_results]
    aug_lex = df_high[lex_changed].copy()
    aug_lex["text_base"] = [lex_texts[i] for i in range(len(lex_texts)) if lex_changed[i]]
    aug_lex["aug_type"] = "lexical"
    if len(aug_lex) > 0:
        aug_rows.append(aug_lex)
    print(f"Aumentación léxica: {len(aug_lex)} filas")

# Concatenar todo
if aug_rows:
    aug_df = pd.concat(aug_rows, ignore_index=True)
    mapped_labels = aug_df["decade"].astype(str).map(label2id)
    missing_labels = mapped_labels.isna().sum()
    if missing_labels > 0:
        print(f"⚠️  Labels missing en aug_df: {missing_labels}")
        print(f"  Valores perdidos: {aug_df.loc[mapped_labels.isna(), 'decade'].unique()}")
    aug_df["label"] = mapped_labels.astype(int)
    train_df = pd.concat([train_df, aug_df], ignore_index=True)
    print(f"Train después de aumentación: {len(train_df)} filas")

train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"Train final: {len(train_df)} filas")


Aumentación OCR: 1930 filas
Aumentación hyphen: 22730 filas
Aumentación masking: 22730 filas
Aumentación léxica: 620 filas
Train después de aumentación: 72670 filas
Train final: 72670 filas


## PASO 7 — Fine-tuning BETO desde `dccuchile/bert-base-spanish-wwm-cased`


In [7]:

# === PASO 7 — Fine-tuning BETO ===

if Path(BETO_FT_CKPT).exists():
    print(f"Checkpoint ya existe: {BETO_FT_CKPT}. Saltando fine-tuning.")
else:
    print("Iniciando fine-tuning de BETO...")
    beto_tok_ft = AutoTokenizer.from_pretrained(BETO_NAME, use_fast=True)
    beto_for_train = AutoModelForSequenceClassification.from_pretrained(
        BETO_NAME,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
    )

    def tok_beto(batch):
        return beto_tok_ft(
            batch["text"],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )

    hf_tr = HFDataset.from_pandas(
        train_df[["text_base", "label"]].rename(columns={"text_base": "text", "label": "labels"}))
    hf_va = HFDataset.from_pandas(
        val_df[["text_base", "label"]].rename(columns={"text_base": "text", "label": "labels"}))
    hf_tr = hf_tr.map(tok_beto, batched=True, remove_columns=["text"])
    hf_va = hf_va.map(tok_beto, batched=True, remove_columns=["text"])

    beto_tr_args = TrainingArguments(
        output_dir=str(ROOT / "ckpt-beto"),
        num_train_epochs=BETO_FT_EPOCHS,
        per_device_train_batch_size=BETO_FT_BATCH_SIZE,
        per_device_eval_batch_size=BETO_FT_BATCH_SIZE,
        learning_rate=BETO_FT_LR,
        weight_decay=0.01,
        warmup_ratio=0.1,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

    beto_tr = Trainer(
        model=beto_for_train,
        args=beto_tr_args,
        train_dataset=hf_tr,
        eval_dataset=hf_va,
        tokenizer=beto_tok_ft,
        data_collator=DataCollatorWithPadding(tokenizer=beto_tok_ft),
        compute_metrics=lambda p: {
            "accuracy": accuracy_score(p.label_ids, p.predictions.argmax(-1))
        },
    )

    beto_tr.train()
    beto_tr.save_model(BETO_FT_CKPT)
    print(f"Checkpoint guardado: {BETO_FT_CKPT}")

    del beto_for_train, beto_tr, hf_tr, hf_va, beto_tok_ft
    gc.collect(); torch.cuda.empty_cache()


Iniciando fine-tuning de BETO...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	

Map:   0%|          | 0/72670 [00:00<?, ? examples/s]

Map:   0%|          | 0/6165 [00:00<?, ? examples/s]

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

## PASO 8 — Fine-tuning RoBERTa-BNE


In [ ]:

# === PASO 8 — Fine-tuning RoBERTa-BNE ===

if Path(ROBERTA_BNE_CKPT).exists():
    print(f"Checkpoint ya existe: {ROBERTA_BNE_CKPT}. Saltando fine-tuning.")
else:
    print("Iniciando fine-tuning de RoBERTa-BNE...")
    roberta_tok_ft = AutoTokenizer.from_pretrained(ROBERTA_BNE_NAME, use_fast=True)
    roberta_for_train = AutoModelForSequenceClassification.from_pretrained(
        ROBERTA_BNE_NAME,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
    )

    def tok_roberta(batch):
        return roberta_tok_ft(
            batch["text"],
            truncation=True,
            max_length=ROBERTA_MAX_LENGTH,
            padding=False,
        )

    hf_tr = HFDataset.from_pandas(
        train_df[["text_base", "label"]].rename(columns={"text_base": "text", "label": "labels"}))
    hf_va = HFDataset.from_pandas(
        val_df[["text_base", "label"]].rename(columns={"text_base": "text", "label": "labels"}))
    hf_tr = hf_tr.map(tok_roberta, batched=True, remove_columns=["text"])
    hf_va = hf_va.map(tok_roberta, batched=True, remove_columns=["text"])

    roberta_tr_args = TrainingArguments(
        output_dir=str(ROOT / "ckpt-roberta-bne"),
        num_train_epochs=ROBERTA_EPOCHS,
        per_device_train_batch_size=ROBERTA_BATCH_SIZE,
        per_device_eval_batch_size=ROBERTA_BATCH_SIZE,
        learning_rate=ROBERTA_LR,
        weight_decay=0.01,
        warmup_ratio=0.1,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

    roberta_tr = Trainer(
        model=roberta_for_train,
        args=roberta_tr_args,
        train_dataset=hf_tr,
        eval_dataset=hf_va,
        tokenizer=roberta_tok_ft,
        data_collator=DataCollatorWithPadding(tokenizer=roberta_tok_ft),
        compute_metrics=lambda p: {
            "accuracy": accuracy_score(p.label_ids, p.predictions.argmax(-1))
        },
    )

    roberta_tr.train()
    roberta_tr.save_model(ROBERTA_BNE_CKPT)
    print(f"Checkpoint guardado: {ROBERTA_BNE_CKPT}")

    del roberta_for_train, roberta_tr, hf_tr, hf_va, roberta_tok_ft
    gc.collect(); torch.cuda.empty_cache()


## PASO 9 — Inferencia BETO (logits + embeddings)


In [ ]:

# === PASO 9a — Inferencia BETO logits ===

def build_chunked_for_inference(df, tokenizer, max_length=MAX_LENGTH, stride=EVAL_STRIDE):
    enc = tokenizer(df["text_base"].tolist(), truncation=True, max_length=max_length,
                    stride=stride, return_overflowing_tokens=True, padding=False)
    doc_ids = np.array(enc.pop("overflow_to_sample_mapping"), dtype=np.int64)
    ds = HFDataset.from_dict({"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]})
    return ds, doc_ids

def avg_logits_by_doc(chunk_logits, doc_ids, n_docs):
    out = np.zeros((n_docs, chunk_logits.shape[1]), dtype=np.float32)
    counts = np.zeros(n_docs, dtype=np.int32)
    for i, d in enumerate(doc_ids):
        out[int(d)] += chunk_logits[i]; counts[int(d)] += 1
    counts = np.maximum(counts, 1)
    return out / counts[:, None]

def make_inference_args(output_dir):
    return TrainingArguments(
        output_dir=str(output_dir), per_device_eval_batch_size=BATCH_SIZE,
        fp16=torch.cuda.is_available(), report_to="none",
        do_train=False, do_eval=False, save_strategy="no", logging_strategy="no",
    )

print(f"Cargando BETO fine-tuneado: {BETO_FT_CKPT}")
beto_tokenizer = AutoTokenizer.from_pretrained(BETO_NAME, use_fast=True)
beto_clf = AutoModelForSequenceClassification.from_pretrained(BETO_FT_CKPT).to(DEVICE).eval()

beto_trainer = Trainer(model=beto_clf, args=make_inference_args(ROOT / "tmp_beto"),
                       data_collator=DataCollatorWithPadding(tokenizer=beto_tokenizer))

print("Inferencia BETO logits (val, eval)...")
val_ds, val_doc   = build_chunked_for_inference(val_df,  beto_tokenizer)
eval_ds, eval_doc = build_chunked_for_inference(eval_df, beto_tokenizer)
val_pred  = beto_trainer.predict(val_ds)
eval_pred = beto_trainer.predict(eval_ds)
val_logits_beto  = avg_logits_by_doc(val_pred.predictions,  val_doc,  n_docs=len(val_df))
eval_logits_beto = avg_logits_by_doc(eval_pred.predictions, eval_doc, n_docs=len(eval_df))
val_probs_beto  = softmax(val_logits_beto,  axis=-1)
eval_probs_beto = softmax(eval_logits_beto, axis=-1)
report("BETO (fine-tuneado)", val_probs_beto)


In [ ]:

# === PASO 9b — Extracción de embeddings BETO ===

@torch.no_grad()
def extract_embeddings(encoder, tokenizer, texts, max_length=MAX_LENGTH, batch_size=BATCH_SIZE):
    encoder.eval()
    hidden = encoder.config.hidden_size
    embs = np.zeros((len(texts), hidden), dtype=np.float32)
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i+batch_size]
        enc = tokenizer(chunk, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(DEVICE)
        out = encoder(**enc)
        last = out.last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1).float()
        pooled = (last * mask).sum(1) / mask.sum(1).clamp(min=1.0)
        embs[i:i+len(chunk)] = pooled.cpu().numpy()
    return embs

print("Extrayendo embeddings de BETO (sobre text_base)...")
beto_encoder = beto_clf.bert  # base BERT model
emb_train = extract_embeddings(beto_encoder, beto_tokenizer, train_df["text_base"].tolist())
emb_val   = extract_embeddings(beto_encoder, beto_tokenizer, val_df["text_base"].tolist())
emb_eval  = extract_embeddings(beto_encoder, beto_tokenizer, eval_df["text_base"].tolist())
print(f"  embeddings: train {emb_train.shape}  val {emb_val.shape}  eval {emb_eval.shape}")

del beto_trainer, beto_clf, beto_encoder, beto_tokenizer, val_ds, eval_ds, val_pred, eval_pred
gc.collect(); torch.cuda.empty_cache()


## PASO 10 — Inferencia RoBERTa-BNE


In [ ]:

# === PASO 10 — Inferencia RoBERTa-BNE ===

if not Path(ROBERTA_BNE_CKPT).exists():
    print(f"Checkpoint no encontrado: {ROBERTA_BNE_CKPT}. Usando modelo base.")
    roberta_ckpt = ROBERTA_BNE_NAME
else:
    roberta_ckpt = ROBERTA_BNE_CKPT

print(f"Cargando RoBERTa-BNE: {roberta_ckpt}")
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_BNE_NAME, use_fast=True)
roberta_clf = AutoModelForSequenceClassification.from_pretrained(
    roberta_ckpt, num_labels=num_labels,
    id2label=id2label, label2id=label2id,
).to(DEVICE).eval()

roberta_infer_args = TrainingArguments(
    output_dir=str(ROOT / "tmp_roberta"), per_device_eval_batch_size=ROBERTA_BATCH_SIZE,
    fp16=torch.cuda.is_available(), report_to="none",
    do_train=False, do_eval=False, save_strategy="no", logging_strategy="no",
)
roberta_trainer_inf = Trainer(
    model=roberta_clf,
    args=roberta_infer_args,
    data_collator=DataCollatorWithPadding(tokenizer=roberta_tokenizer),
)

print("Inferencia RoBERTa-BNE logits (val, eval)...")
val_ds_rb, val_doc_rb   = build_chunked_for_inference(
    val_df,  roberta_tokenizer, max_length=ROBERTA_MAX_LENGTH, stride=ROBERTA_EVAL_STRIDE)
eval_ds_rb, eval_doc_rb = build_chunked_for_inference(
    eval_df, roberta_tokenizer, max_length=ROBERTA_MAX_LENGTH, stride=ROBERTA_EVAL_STRIDE)

val_pred_rb  = roberta_trainer_inf.predict(val_ds_rb)
eval_pred_rb = roberta_trainer_inf.predict(eval_ds_rb)

val_logits_roberta  = avg_logits_by_doc(val_pred_rb.predictions,  val_doc_rb,  n_docs=len(val_df))
eval_logits_roberta = avg_logits_by_doc(eval_pred_rb.predictions, eval_doc_rb, n_docs=len(eval_df))
val_probs_roberta  = softmax(val_logits_roberta,  axis=-1)
eval_probs_roberta = softmax(eval_logits_roberta, axis=-1)
report("RoBERTa-BNE", val_probs_roberta)

del roberta_trainer_inf, roberta_clf, val_ds_rb, eval_ds_rb, val_pred_rb, eval_pred_rb, roberta_tokenizer
gc.collect(); torch.cuda.empty_cache()


## PASO 11 — Features manuales (29d) + StandardScaler


In [ ]:

# === PASO 11 — Features manuales + StandardScaler ===

def extract_all_features(df, feature_cols):
    return np.stack([df[c].values for c in feature_cols], axis=1).astype(np.float32)

X_hc_train_raw = extract_all_features(train_df, FEATURE_COLUMNS)
X_hc_val_raw   = extract_all_features(val_df,   FEATURE_COLUMNS)
X_hc_eval_raw  = extract_all_features(eval_df,  FEATURE_COLUMNS)

print(f"Features raw: train {X_hc_train_raw.shape}")

scaler_hc = StandardScaler()
X_hc_train = scaler_hc.fit_transform(X_hc_train_raw)
X_hc_val   = scaler_hc.transform(X_hc_val_raw)
X_hc_eval  = scaler_hc.transform(X_hc_eval_raw)

joblib.dump(scaler_hc, ROOT / "feature_scaler.pkl")
print(f"Scaler guardado en {ROOT / 'feature_scaler.pkl'}")

# Normalizar embeddings también
emb_mean = emb_train.mean(0, keepdims=True)
emb_std  = emb_train.std(0, keepdims=True) + 1e-6
emb_train_n = (emb_train - emb_mean) / emb_std
emb_val_n   = (emb_val   - emb_mean) / emb_std
emb_eval_n  = (emb_eval  - emb_mean) / emb_std

# Concatenar para MLP
X_mlp_train = np.hstack([emb_train_n, X_hc_train]).astype(np.float32)
X_mlp_val   = np.hstack([emb_val_n,   X_hc_val]).astype(np.float32)
X_mlp_eval  = np.hstack([emb_eval_n,  X_hc_eval]).astype(np.float32)
MLP_INPUT_DIM = X_mlp_train.shape[1]
print(f"MLP input dims: {MLP_INPUT_DIM} (embeddings {emb_train_n.shape[1]}d + features {X_hc_train.shape[1]}d)")


## PASO 12 — MLP (embeddings 768d + features 29d = 797d)


In [ ]:

# === PASO 12 — MLP sobre [embeddings BETO + features] ===

class MLP(nn.Module):
    def __init__(self, in_dim, num_labels, h1=512, h2=256, p=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, h1), nn.BatchNorm1d(h1), nn.ReLU(), nn.Dropout(p),
            nn.Linear(h1, h2),     nn.BatchNorm1d(h2), nn.ReLU(), nn.Dropout(p * 0.75),
            nn.Linear(h2, num_labels),
        )
    def forward(self, x):
        return self.net(x)

def train_torch_classifier(model, X_tr, y_tr, X_va, y_va, epochs=60, lr=1e-3, bs=256, patience=8):
    model = model.to(DEVICE)
    Xtr = torch.tensor(X_tr); ytr = torch.tensor(y_tr, dtype=torch.long)
    Xva = torch.tensor(X_va).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    soft = SOFT_TARGETS_T.to(DEVICE)
    n = len(Xtr)
    best_acc, best_state, no_improve = -1, None, 0
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            xb = Xtr[idx].to(DEVICE); yb = ytr[idx].to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = F.kl_div(F.log_softmax(logits, -1), soft[yb], reduction="batchmean")
            loss.backward(); opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            va_logits = model(Xva).cpu().numpy()
        va_acc = accuracy_score(y_va, np.argmax(va_logits, -1))
        if va_acc > best_acc:
            best_acc = va_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_acc

@torch.no_grad()
def predict_torch(model, X):
    model.eval()
    out = model(torch.tensor(X).to(DEVICE)).cpu().numpy()
    return softmax(out, axis=-1)

print("Entrenando MLP...")
mlp = MLP(MLP_INPUT_DIM, num_labels)
mlp, mlp_best_acc = train_torch_classifier(
    mlp, X_mlp_train, train_df["label"].to_numpy(), X_mlp_val, val_true_lbl)
val_proba_mlp  = predict_torch(mlp, X_mlp_val)
eval_proba_mlp = predict_torch(mlp, X_mlp_eval)
report("MLP (emb+29features)", val_proba_mlp)


## PASO 13 — CharCNN (vocab histórico, text_base)


In [ ]:

# === PASO 13 — CharCNN sobre text_base ===

CHAR_MAXLEN = 600
all_chars = Counter()
for t in train_df["text_base"]:
    all_chars.update(t.lower())
vocab_chars = ["<pad>", "<unk>"] + [c for c, cnt in all_chars.most_common() if cnt >= 5]
char2idx = {c: i for i, c in enumerate(vocab_chars)}
VOCAB_SIZE = len(vocab_chars)
print(f"Vocab de caracteres (sobre text_base): {VOCAB_SIZE}")

has_long_s = any("ſ" in c for c in vocab_chars)
print(f"Contiene ſ (s larga): {has_long_s}")

def encode_chars(text, maxlen=CHAR_MAXLEN):
    t = text.lower()[:maxlen]
    ids = [char2idx.get(c, 1) for c in t]
    if len(ids) < maxlen:
        ids = ids + [0] * (maxlen - len(ids))
    return np.array(ids, dtype=np.int64)

X_cnn_train = np.stack([encode_chars(t) for t in train_df["text_base"]])
X_cnn_val   = np.stack([encode_chars(t) for t in val_df["text_base"]])
X_cnn_eval  = np.stack([encode_chars(t) for t in eval_df["text_base"]])
print(f"  char-encoded: {X_cnn_train.shape}")

class CharCNN(nn.Module):
    def __init__(self, vocab_size, num_labels, emb_dim=64, n_filters=128,
                 kernel_sizes=(3, 4, 5, 6), p=0.4):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(emb_dim, n_filters, k, padding=k // 2) for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(p)
        self.fc = nn.Sequential(
            nn.Linear(n_filters * len(kernel_sizes), 256), nn.ReLU(), nn.Dropout(p),
            nn.Linear(256, num_labels),
        )
    def forward(self, x):
        e = self.emb(x).transpose(1, 2)
        feats = []
        for conv in self.convs:
            c = F.relu(conv(e))
            c = F.max_pool1d(c, c.size(2)).squeeze(2)
            feats.append(c)
        h = torch.cat(feats, dim=1)
        h = self.dropout(h)
        return self.fc(h)

def train_cnn(model, X_tr, y_tr, X_va, y_va, epochs=40, lr=1e-3, bs=128, patience=6):
    model = model.to(DEVICE)
    Xtr = torch.tensor(X_tr); ytr = torch.tensor(y_tr, dtype=torch.long)
    Xva = torch.tensor(X_va).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    soft = SOFT_TARGETS_T.to(DEVICE)
    n = len(Xtr)
    best_acc, best_state, no_improve = -1, None, 0
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            xb = Xtr[idx].to(DEVICE); yb = ytr[idx].to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = F.kl_div(F.log_softmax(logits, -1), soft[yb], reduction="batchmean")
            loss.backward(); opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            preds = []
            for i in range(0, len(Xva), 256):
                preds.append(model(Xva[i:i+256]).cpu().numpy())
            va_logits = np.vstack(preds)
        va_acc = accuracy_score(y_va, np.argmax(va_logits, -1))
        if va_acc > best_acc:
            best_acc = va_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break
        if ep % 5 == 0:
            print(f"  epoch {ep}: val_acc={va_acc:.4f} (best={best_acc:.4f})")
    model.load_state_dict(best_state)
    return model, best_acc

@torch.no_grad()
def predict_cnn(model, X):
    model.eval()
    preds = []
    Xt = torch.tensor(X).to(DEVICE)
    for i in range(0, len(Xt), 256):
        preds.append(model(Xt[i:i+256]).cpu().numpy())
    return softmax(np.vstack(preds), axis=-1)

print("Entrenando CharCNN...")
cnn = CharCNN(VOCAB_SIZE, num_labels)
cnn, cnn_best_acc = train_cnn(cnn, X_cnn_train, train_df["label"].to_numpy(), X_cnn_val, val_true_lbl)
val_proba_cnn  = predict_cnn(cnn, X_cnn_val)
eval_proba_cnn = predict_cnn(cnn, X_cnn_eval)
report("CharCNN (text_base)", val_proba_cnn)

del cnn
gc.collect(); torch.cuda.empty_cache()


## PASO 14 — TF-IDF + modelos clásicos


In [ ]:

# === PASO 14 — TF-IDF + LogReg, LinearSVC calibrado, Ridge ordinal ===

print("Vectorizando TF-IDF char_wb 2-5 sobre text_norm_soft...")
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2,
                             max_features=250_000, sublinear_tf=True)
X_tf_train = vectorizer.fit_transform(train_df["text_norm_soft"])
X_tf_val   = vectorizer.transform(val_df["text_norm_soft"])
X_tf_eval  = vectorizer.transform(eval_df["text_norm_soft"])
y_train = train_df["label"].to_numpy()

# (5) LogReg
print("LogReg...")
logreg = LogisticRegression(max_iter=2000, n_jobs=-1, C=1.0, class_weight="balanced")
logreg.fit(X_tf_train, y_train)
val_proba_lr  = align_proba(logreg.predict_proba(X_tf_val),  logreg.classes_, num_labels)
eval_proba_lr = align_proba(logreg.predict_proba(X_tf_eval), logreg.classes_, num_labels)
report("TF-IDF + LogReg (soft)", val_proba_lr)

# (6) LinearSVC calibrado
print("LinearSVC calibrado (~5 min)...")
svc = CalibratedClassifierCV(LinearSVC(C=0.5, class_weight="balanced", max_iter=3000, random_state=SEED),
                             method="sigmoid", cv=3)
svc.fit(X_tf_train, y_train)
val_proba_svc  = align_proba(svc.predict_proba(X_tf_val),  svc.classes_, num_labels)
eval_proba_svc = align_proba(svc.predict_proba(X_tf_eval), svc.classes_, num_labels)
report("LinearSVC (calibrado, soft)", val_proba_svc)

# (7) Regresión ordinal: Ridge sobre década numérica
print("Regresión ordinal (Ridge sobre década)...")
y_train_dec = train_df["decade"].to_numpy().astype(np.float32)
ridge = Ridge(alpha=1.0)
ridge.fit(X_tf_train, y_train_dec)

def reg_to_proba(reg_pred, labels, sigma=1.0):
    labels_arr = np.array(labels, dtype=np.float32)
    probs = np.zeros((len(reg_pred), len(labels)), dtype=np.float32)
    for i, p in enumerate(reg_pred):
        w = np.exp(-((labels_arr - p) ** 2) / (2.0 * sigma ** 2))
        probs[i] = w / w.sum()
    return probs

val_reg_pred  = ridge.predict(X_tf_val)
eval_reg_pred = ridge.predict(X_tf_eval)
val_proba_reg  = reg_to_proba(val_reg_pred,  labels, sigma=1.0)
eval_proba_reg = reg_to_proba(eval_reg_pred, labels, sigma=1.0)
report("Regresión ordinal (Ridge)", val_proba_reg)


## PASO 15 — Ablation (RandomForest sobre features)


In [ ]:

# === PASO 15 — Ablation: RandomForest sobre grupos de features ===

def evaluate_ablation(train_df, val_df, config_name, feature_cols):
    X_tr = train_df[feature_cols].fillna(0).values
    X_va = val_df[feature_cols].fillna(0).values
    y_tr = train_df["label"].values
    y_va = val_df["label"].values
    clf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED)
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_va)
    return {
        "config": config_name,
        "accuracy": round(accuracy_score(y_va, y_pred), 4),
        "macro_f1": round(f1_score(y_va, y_pred, average="macro"), 4),
        "n_features": len(feature_cols),
    }

base_feats = ["num_chars", "num_words"]
stylo_feats = [c for c in FEATURE_COLUMNS if c in [
    "semicolon_rate", "emdash_rate", "colon_rate", "comma_rate", "sentence_len_mean"]]
hist_feats = ["obsolete_accent_rate", "long_s_proxy_rate", "latin_abbrev_rate", "honorific_caps_rate"]
ocr_feats = ["ocr_noise_score", "ocr_symbol_rate", "oov_rate", "corrupt_token_rate"]
syn_feats = ["ttr", "subordination_rate", "historic_stopword_rate", "clause_len_mean", "citation_pattern_rate"]

ablations = []
ablations.append(evaluate_ablation(train_df, val_df, "baseline", base_feats))
ablations.append(evaluate_ablation(train_df, val_df, "+stylometry", base_feats + stylo_feats))
ablations.append(evaluate_ablation(train_df, val_df, "+historical_spelling", base_feats + hist_feats))
ablations.append(evaluate_ablation(train_df, val_df, "+ocr_features", base_feats + ocr_feats))
ablations.append(evaluate_ablation(train_df, val_df, "+syntax", base_feats + syn_feats))
ablations.append(evaluate_ablation(train_df, val_df, "all_features", FEATURE_COLUMNS))

df_ablation = pd.DataFrame(ablations)
print("\n=== ABLATION STUDY ===")
print(df_ablation.to_string(index=False))
print("\n--- Ganancia incremental ---")
for i in range(1, len(df_ablation)):
    prev = df_ablation.iloc[i-1]
    curr = df_ablation.iloc[i]
    print(f"{curr['config']}: acc +{curr['accuracy'] - prev['accuracy']:.4f}, "
          f"f1 +{curr['macro_f1'] - prev['macro_f1']:.4f}")


## PASO 16 — Ensemble voting (7 modelos)


In [ ]:

# === PASO 16 — Ensemble (7 modelos): soft / hard / weighted voting ===

all_models = {
    "BETO":           (val_probs_beto, eval_probs_beto),
    "RoBERTa-BNE":    (val_probs_roberta, eval_probs_roberta),
    "CharCNN":        (val_proba_cnn,  eval_proba_cnn),
    "MLP":            (val_proba_mlp,  eval_proba_mlp),
    "TF-IDF LogReg":  (val_proba_lr,   eval_proba_lr),
    "LinearSVC":      (val_proba_svc,  eval_proba_svc),
    "Reg. ordinal":   (val_proba_reg,  eval_proba_reg),
}

print("=== Individuales en val ===")
indiv_acc = {}
for name, (vp, _) in all_models.items():
    indiv_acc[name] = report(name, vp)

model_names = list(all_models.keys())
val_probs_all  = [all_models[n][0] for n in model_names]
eval_probs_all = [all_models[n][1] for n in model_names]

# Soft voting uniforme
soft_val  = np.mean(val_probs_all,  axis=0)
soft_eval = np.mean(eval_probs_all, axis=0)
acc_soft = report("SOFT voting (uniforme)", soft_val)

# Hard voting (BETO desempata; es índice 0)
def hard_vote(probs_list, tiebreak_idx=0):
    preds = np.stack([np.argmax(p, axis=-1) for p in probs_list], axis=1)
    out = np.zeros(preds.shape[0], dtype=int)
    for i in range(preds.shape[0]):
        votes = Counter(preds[i])
        top = votes.most_common()
        winners = [c for c, cnt in top if cnt == top[0][1]]
        out[i] = winners[0] if len(winners) == 1 else preds[i, tiebreak_idx]
    return out

hard_pred = hard_vote(val_probs_all, tiebreak_idx=0)
acc_hard = accuracy_score(val_true_lbl, hard_pred)
mae_hard = mean_absolute_error(val_true_dec, [labels[i] for i in hard_pred])
print(f"  [{'HARD voting':32s}] val acc={acc_hard:.4f}  mae={mae_hard:.4f}")

# Weighted soft voting (grid)
print("\nGrid de pesos (weighted soft voting)...")
best = None
step_options = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
for w in itertools.product(step_options, repeat=len(model_names)):
    if abs(sum(w) - 1.0) > 0.01:
        continue
    probs = sum(wi * vi for wi, vi in zip(w, val_probs_all))
    acc = accuracy_score(val_true_lbl, np.argmax(probs, axis=-1))
    if best is None or acc > best[1]:
        best = (w, acc)
w_best, acc_weighted = best
print(f"  mejor: {dict(zip(model_names, [round(x,2) for x in w_best]))}")
print(f"  [{'WEIGHTED soft voting':32s}] val acc={acc_weighted:.4f}")


## PASO 17 — Submission CSV


In [ ]:

# === PASO 17 — Elegir mejor esquema + guardar submission ===

schemes = {"soft_uniform": acc_soft, "hard_voting": acc_hard, "weighted_soft": acc_weighted}
print("=== Esquemas (val acc) ===")
for name, acc in sorted(schemes.items(), key=lambda x: -x[1]):
    print(f"  {name:18s}: {acc:.4f}")

best_scheme = max(schemes, key=schemes.get)
best_indiv  = max(indiv_acc, key=indiv_acc.get)
print(f"\nMejor esquema:    {best_scheme}  ({schemes[best_scheme]:.4f})")
print(f"Mejor individual: {best_indiv}  ({indiv_acc[best_indiv]:.4f})")

# Generar eval del mejor esquema
if best_scheme == "soft_uniform":
    eval_pred_ids = np.argmax(soft_eval, axis=-1)
elif best_scheme == "hard_voting":
    eval_pred_ids = hard_vote(eval_probs_all, tiebreak_idx=0)
else:
    eval_final = sum(wi * ei for wi, ei in zip(w_best, eval_probs_all))
    eval_pred_ids = np.argmax(eval_final, axis=-1)

answers = [id2label[int(i)] for i in eval_pred_ids]

# Verificar IDs del submission
assert len(answers) == len(eval_df),     f"ERROR: {len(answers)} predicciones vs {len(eval_df)} IDs en eval"

submission = pd.DataFrame({"id": eval_df["id"].to_numpy(), "answer": answers})
assert set(submission["id"]) == set(eval_df["id"]),     "ERROR: IDs del submission no coinciden con eval.csv"
print(f"✓ Submission válido: {len(submission)} filas, IDs correctos")
print(submission.head())

ts = datetime.now().strftime("%Y%m%d_%H%M")
out_name = ROOT / f"submission_v81_{best_scheme}_{ts}.csv"
submission.to_csv(out_name, index=False)
submission.to_csv(ROOT / "submission.csv", index=False)
print(f"\nsaved {out_name} {submission.shape}")

# Guardar todas las predicciones
np.savez(ROOT / "predictions_v81.npz",
    **{f"val_{n}": all_models[n][0] for n in model_names},
    **{f"eval_{n}": all_models[n][1] for n in model_names},
    val_true_lbl=val_true_lbl, val_true_dec=val_true_dec,
)
print("predicciones guardadas en predictions_v81.npz")

if schemes[best_scheme] <= indiv_acc[best_indiv] + 0.003:
    print("\n⚠️  La votación no supera claramente al mejor modelo individual (dentro del ruido).")
else:
    print(f"\n✓ La votación mejora sobre el mejor individual por {schemes[best_scheme]-indiv_acc[best_indiv]:.4f}.")
